# Semana 4: Sistemas de Ecuaciones y la Base de la Regresión
## Notebook Conceptual (NB1) - Resolviendo Sistemas con Datos Sintéticos

### Propósito de la sesión
Conectar la resolución de sistemas lineales con el primer modelo de machine learning: la **regresión lineal**. Exploraremos sistemas cuadrados (solución única) y sistemas sobredeterminados (más ecuaciones que incógnitas), donde la solución por **mínimos cuadrados** se convierte en la herramienta fundamental.

### Objetivos de aprendizaje
*   Representar sistemas de ecuaciones lineales en forma matricial $\mathbf{Ax} = \mathbf{b}$.
*   Resolver sistemas cuadrados usando **eliminación gaussiana** (implementación simple) y `np.linalg.solve`.
*   Plantear y resolver sistemas sobredeterminados mediante la **ecuación normal**.
*   Interpretar geométricamente la solución de mínimos cuadrados como una proyección.
*   Conectar estos conceptos con la **regresión lineal**, el MSE y aplicaciones en robótica y visión.

## Configuración Inicial

Importamos las librerías necesarias y generamos datos sintéticos.

In [ ]:
# Importamos librerías
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D

# Configuración de estilo
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 5)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. Representación Matricial de Sistemas Lineales

Un sistema de ecuaciones lineales puede escribirse como:

$$\mathbf{Ax} = \mathbf{b}$$

donde:
*   $\mathbf{A} \in \mathbb{R}^{m \times n}$ es la matriz de coeficientes,
*   $\mathbf{x} \in \mathbb{R}^{n}$ es el vector de incógnitas,
*   $\mathbf{b} \in \mathbb{R}^{m}$ es el vector de términos independientes.

### Ejemplo 2x2:
$$
\begin{cases}
2x + 3y = 8 \\
4x - y = 2
\end{cases}
\quad \Rightarrow \quad
\begin{bmatrix} 2 & 3 \\ 4 & -1 \end{bmatrix}
\begin{bmatrix} x \\ y \end{bmatrix}
=
\begin{bmatrix} 8 \\ 2 \end{bmatrix}
$$

In [ ]:
# Representación en NumPy
A = np.array([[2, 3],
              [4, -1]])
b = np.array([8, 2])

print("Matriz A:")
print(A)
print("\nVector b:", b)
print(f"\nDimensiones: A {A.shape}, b {b.shape}")

---
## 2. Sistemas Cuadrados (m = n) - Solución Única

Cuando el número de ecuaciones es igual al número de incógnitas y la matriz $\mathbf{A}$ es invertible, existe una única solución.

### 2.1. Eliminación Gaussiana (implementación simple)

Implementaremos una versión simplificada de eliminación gaussiana para sistemas 2x2 y 3x3.

In [ ]:
def eliminacion_gaussiana_2x2(A, b):
    """Resuelve un sistema 2x2 mediante eliminación gaussiana simple."""
    a11, a12 = A[0, 0], A[0, 1]
    a21, a22 = A[1, 0], A[1, 1]
    b1, b2 = b[0], b[1]
    
    # Hacemos cero el elemento (2,1)
    factor = a21 / a11
    a22_nuevo = a22 - factor * a12
    b2_nuevo = b2 - factor * b1
    
    # Sustitución regresiva
    y = b2_nuevo / a22_nuevo
    x = (b1 - a12 * y) / a11
    
    return np.array([x, y])

# Resolvemos el sistema 2x2
sol_2x2 = eliminacion_gaussiana_2x2(A, b)
print(f"🔷 Solución del sistema 2x2 (eliminación gaussiana): x = {sol_2x2[0]:.2f}, y = {sol_2x2[1]:.2f}")

# Verificación
print(f"Verificación: A·x = {A @ sol_2x2}, b = {b}")

### 2.2. Sistema 3x3 - Eliminación Gaussiana (genérica)

Implementamos una versión más general para sistemas 3x3.

In [ ]:
def eliminacion_gaussiana_3x3(A, b):
    """Resuelve un sistema 3x3 mediante eliminación gaussiana con sustitución regresiva."""
    # Creamos la matriz aumentada
    M = np.column_stack([A, b]).astype(float)
    
    # Forward elimination
    for i in range(3):
        # Pivote
        pivot = M[i, i]
        # Hacemos ceros debajo del pivote
        for j in range(i+1, 3):
            factor = M[j, i] / pivot
            M[j, i:] -= factor * M[i, i:]
    
    # Sustitución regresiva
    x = np.zeros(3)
    for i in range(2, -1, -1):
        x[i] = (M[i, 3] - np.dot(M[i, i+1:3], x[i+1:3])) / M[i, i]
    
    return x

# Creamos un sistema 3x3 con solución conocida
A3 = np.array([[2, 1, -1],
               [-3, -1, 2],
               [-2, 1, 2]], dtype=float)
b3 = np.array([8, -11, -3], dtype=float)

sol_3x3 = eliminacion_gaussiana_3x3(A3, b3)
print(f"🔷 Solución del sistema 3x3: {sol_3x3}")
print(f"Verificación: A·x = {A3 @ sol_3x3}")
print(f"b original: {b3}")

### 2.3. Resolución con `np.linalg.solve`

NumPy proporciona una función optimizada que utiliza factorización LU con pivotamiento.

In [ ]:
# Para el sistema 2x2
sol_np_2x2 = np.linalg.solve(A, b)
print(f"🔶 np.linalg.solve (2x2): {sol_np_2x2}")

# Para el sistema 3x3
sol_np_3x3 = np.linalg.solve(A3, b3)
print(f"🔶 np.linalg.solve (3x3): {sol_np_3x3}")

# Comparación con nuestra implementación
print(f"\n📊 Diferencia entre métodos (3x3): {np.max(np.abs(sol_3x3 - sol_np_3x3)):.2e}")

---
## 3. Sistemas Sobredeterminados (m > n) - Mínimos Cuadrados

En machine learning, normalmente tenemos más ecuaciones (datos) que incógnitas (parámetros). El sistema $\mathbf{X}\beta \approx \mathbf{y}$ no tiene solución exacta, así que buscamos la mejor solución en el sentido de minimizar la norma del error.

### 3.1. Planteamiento del problema

Dado $\mathbf{X} \in \mathbb{R}^{m \times n}$ (con $m > n$) y $\mathbf{y} \in \mathbb{R}^m$, queremos encontrar $\beta$ que minimice:

$$\|\mathbf{X}\beta - \mathbf{y}\|_2^2 = \sum_{i=1}^m (\hat{y}_i - y_i)^2$$

### 3.2. Ecuación Normal

La solución analítica (si $\mathbf{X}^T\mathbf{X}$ es invertible) es:

$$\hat{\beta} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

### Ejemplo: Ajuste de una recta por mínimos cuadrados

Queremos ajustar una recta $y = \beta_0 + \beta_1 x$ a los puntos $(x_i, y_i)$. El sistema es:

$$
\begin{pmatrix}
1 & x_1 \\
1 & x_2 \\
\vdots & \vdots \\
1 & x_m
\end{pmatrix}
\begin{pmatrix} \beta_0 \\ \beta_1 \end{pmatrix}
\approx
\begin{pmatrix} y_1 \\ y_2 \\ \vdots \\ y_m \end{pmatrix}
$$

In [ ]:
# Generamos datos sintéticos con ruido
m = 20  # número de puntos (ecuaciones)
x = np.linspace(0, 10, m)
beta_true = np.array([2.5, 1.2])  # β0, β1 verdaderos
y_true = beta_true[0] + beta_true[1] * x
y = y_true + np.random.randn(m) * 2  # añadimos ruido

# Construimos la matriz X (con columna de unos para el intercepto)
X = np.column_stack([np.ones(m), x])

print(f"Shape de X: {X.shape} (20 ecuaciones, 2 incógnitas)")
print(f"Shape de y: {y.shape}")

In [ ]:
# Resolvemos usando la ecuación normal
XTX = X.T @ X
XTy = X.T @ y
beta_ols = np.linalg.solve(XTX, XTy)

print(f"🔷 Coeficientes estimados por ecuación normal:")
print(f"   β₀ (intercepto) = {beta_ols[0]:.4f}")
print(f"   β₁ (pendiente)   = {beta_ols[1]:.4f}")
print(f"Coeficientes verdaderos: β₀={beta_true[0]}, β₁={beta_true[1]}")

# Calculamos el error cuadrático medio
y_pred = X @ beta_ols
mse = np.mean((y - y_pred)**2)
print(f"\n📉 MSE: {mse:.4f}")

# Visualizamos el ajuste
plt.figure(figsize=(10, 6))
plt.scatter(x, y, label='Datos con ruido', alpha=0.7)
plt.plot(x, y_true, 'g--', label='Recta verdadera', linewidth=2)
plt.plot(x, y_pred, 'r-', label='Ajuste mínimos cuadrados', linewidth=2)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Ajuste de una recta por mínimos cuadrados')
plt.legend()
plt.grid(True)
plt.show()

### 3.3. Interpretación geométrica

La solución de mínimos cuadrados proyecta el vector $\mathbf{y}$ sobre el espacio columna de $\mathbf{X}$. El residuo $\mathbf{r} = \mathbf{y} - \mathbf{X}\beta$ es ortogonal a dicho espacio.

Visualicemos esto en 3D para un caso simple con 2 variables (3 puntos).

In [ ]:
# Caso simple: 3 puntos, queremos ajustar un plano (con intercepto y 1 variable)
# En realidad, con 3 puntos y 2 parámetros, el sistema sería determinado (solución exacta)
# pero lo usaremos para visualizar la idea de espacio columna.

# Tomamos 3 puntos
x_3d = np.array([1, 2, 3])
y_3d = np.array([2.1, 2.9, 4.2])
X_3d = np.column_stack([np.ones(3), x_3d])

# Resolvemos (sistema cuadrado 3x3, pero X no es cuadrada, usamos lstsq)
beta_3d, _, _, _ = np.linalg.lstsq(X_3d, y_3d, rcond=None)

# Proyección
y_proj = X_3d @ beta_3d

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Dibujamos los puntos originales (en el plano x-y, pero con z=y)
ax.scatter(X_3d[:, 1], np.zeros(3), y_3d, c='red', s=100, label='Puntos (x, y)')

# Dibujamos los puntos proyectados (sobre el plano de regresión)
ax.scatter(X_3d[:, 1], np.zeros(3), y_proj, c='blue', s=100, label='Proyección')

# Dibujamos el plano de regresión (que en este caso es una línea en 2D, pero en 3D es un plano)
# El espacio columna de X es un plano en R^3 generado por las columnas de X.
# Para visualizarlo, dibujamos la recta de regresión en el espacio.
xx = np.linspace(0, 4, 10)
yy = beta_3d[0] + beta_3d[1] * xx
ax.plot(xx, np.zeros(10), yy, 'g-', linewidth=2, label='Espacio columna (recta)')

# Dibujamos los residuos (líneas verticales)
for i in range(3):
    ax.plot([X_3d[i, 1], X_3d[i, 1]], [0, 0], [y_3d[i], y_proj[i]], 'k--', alpha=0.5)

ax.set_xlabel('x')
ax.set_ylabel('y (dimensión artificial)')
ax.set_zlabel('z (valor de y)')
ax.set_title('Interpretación geométrica de mínimos cuadrados')
ax.legend()
plt.show()

---
## 4. Conexiones IA: Aplicaciones Reales

### 4.1. Machine Learning: Regresión Lineal
La regresión lineal es el modelo fundacional. La función de coste MSE es exactamente el criterio de mínimos cuadrados.

$$J(\beta) = \frac{1}{n} \sum_{i=1}^n (y_i - \hat{y}_i)^2$$

### 4.2. Robótica: Cinemática
Los sistemas de ecuaciones lineales modelan la cinemática de brazos robóticos (relación entre ángulos articulares y posición del efector final).

### 4.3. Visión por Computador: Transformaciones Geométricas
Las transformaciones (rotación, traslación, escalado) se representan como sistemas lineales.

### 4.4. Optimización
Muchos problemas de optimización se reducen a resolver sistemas lineales (ej. método de Newton).

In [ ]:
# Ejemplo: Transformación geométrica (rotación en 2D)
def rotacion(theta):
    """Matriz de rotación 2D."""
    return np.array([[np.cos(theta), -np.sin(theta)],
                     [np.sin(theta), np.cos(theta)]])

# Punto original
p = np.array([2, 1])

# Rotación de 90 grados
theta = np.pi/2
R = rotacion(theta)
p_rot = R @ p

print(f"🔷 Transformación geométrica (rotación):")
print(f"Punto original: {p}")
print(f"Matriz de rotación {theta:.2f} rad:\n{R}")
print(f"Punto rotado: {p_rot}")

# Visualización
plt.figure(figsize=(6, 6))
plt.quiver(0, 0, p[0], p[1], angles='xy', scale_units='xy', scale=1, color='blue', label='Original')
plt.quiver(0, 0, p_rot[0], p_rot[1], angles='xy', scale_units='xy', scale=1, color='red', label='Rotado')
plt.xlim(-3, 3)
plt.ylim(-3, 3)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Rotación como transformación lineal')
plt.legend()
plt.grid(True)
plt.axhline(0, color='black', linewidth=0.5)
plt.axvline(0, color='black', linewidth=0.5)
plt.axis('equal')
plt.show()

---
## 5. Ejercicios para el Estudiante

1.  **Sistema 2x2 manual:** Resuelve el siguiente sistema usando eliminación gaussiana (a mano o con código) y verifica con `np.linalg.solve`:
    $$
    \begin{cases}
    3x + 4y = 10 \\
    2x - y = 3
    \end{cases}
    $$

2.  **Sistema sobredeterminado:** Genera 50 puntos $(x_i, y_i)$ con $y_i = 3 - 2x_i + \epsilon$, donde $\epsilon$ es ruido normal. Ajusta un modelo lineal $y = \beta_0 + \beta_1 x$ usando la ecuación normal. Compara los coeficientes estimados con los verdaderos.

3.  **MSE y regularización:** Para el ajuste anterior, calcula el MSE. Si añadiéramos regularización L2 (Ridge), ¿cómo esperas que cambien los coeficientes?

4.  **Aplicación en robótica:** Un brazo robótico de 2 grados de libertad tiene una cinemática directa dada por:
    $$
    x = L_1 \cos(\theta_1) + L_2 \cos(\theta_1 + \theta_2) \\
    y = L_1 \sin(\theta_1) + L_2 \sin(\theta_1 + \theta_2)
    $$
    Para pequeños desplazamientos, esto se puede linealizar. Investiga qué significa esto y por qué los sistemas lineales son importantes en robótica.

5.  **Reflexión:** ¿Por qué la ecuación normal $(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$ puede ser numéricamente inestable? ¿Qué alternativa ofrecen `np.linalg.lstsq` o `sklearn`?

---
## Conclusiones de la Sesión

*   Los **sistemas de ecuaciones lineales** $\mathbf{Ax} = \mathbf{b}$ son la base de muchos problemas en ciencia e ingeniería.
*   Para **sistemas cuadrados** con solución única, métodos como eliminación gaussiana o `np.linalg.solve` proporcionan la respuesta exacta.
*   En **sistemas sobredeterminados** (más ecuaciones que incógnitas), la **solución por mínimos cuadrados** minimiza el error cuadrático y es la base de la **regresión lineal**.
*   La **ecuación normal** $\hat{\beta} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$ proporciona una solución analítica, aunque puede ser inestable numéricamente para matrices mal condicionadas.
*   Estos conceptos tienen aplicaciones directas en **machine learning** (regresión), **robótica** (cinemática) y **visión por computador** (transformaciones geométricas).

¡Has conectado el álgebra lineal con tu primer modelo de machine learning!